# 02 · LoRA fine-tuning + test predictions
Trains LoRA adapters on the train split (10% of train stories held out as dev for checkpoint choice), then answers the 668 test items of dataset v4 (534 answerable, 134 with the target "Bilmiyorum.").
Output in Drive, `runs/<RUN_ID>/`: `adapter/` (LoRA weights + config), `run.json` (settings, dev ids, loss history, versions), `loss.png`, `predictions_test.csv`.

Defaults: r=16, alpha=32, dropout=0.05, all linear layers, lr 2e-4 cosine, 3 epochs, batch 8 x 2, loss on the answer only.

**Training size.** `TRAIN_PCT` in the Full training cell sets the share of train stories used (1 to 100). The dev set is the same for every value, and the subsets are nested (the stories in a 25% run are also in the 50% and 100% runs), so runs of different sizes differ only by the added data.

**What the model learns.** 20% of the train items have the target `Bilmiyorum.` (questions part 2 cannot answer). The prompt (`p2`) tells the model to write only "Bilmiyorum." when part 2 gives too few clues. Loss is on the answer only, so the model learns both what to answer and when to abstain.

**Runtime (dataset v4, 100%).** About 8,860 train items, about 1,660 optimizer steps. L4 about 65 min, A100 about 30 min, T4 over 2.5 h. Time scales roughly linearly with `TRAIN_PCT`.

**Resume after a disconnect.** Checkpoints are written to the local disk and copied to Drive (`ckpt/<RUN_ID>/`) every `SAVE_EVERY` steps. If the session drops: reconnect, run every cell in **Setup** and **Code** (skip the smoke test), then run **Full training** again with the same settings. It continues from the last checkpoint on Drive. Finished steps are skipped: a completed training is not repeated and saved predictions are not recomputed. The Drive checkpoints are deleted once the adapter is saved.

## Setup

In [ ]:
# ---- Settings (edit once) ----
DRIVE_ROOT = "/content/drive/MyDrive/gemma-tr-temporal-inference"
LOCAL = "/content/data"            # local copy of the dataset (Drive I/O is slow)
CKPT_LOCAL = "/content/ckpt"       # checkpoints are written here first (fast)
CKPT_DRIVE = f"{DRIVE_ROOT}/ckpt"  # ... and copied here so a new session can resume

In [ ]:
# ---- Mount Drive ----
from google.colab import drive, userdata
drive.mount("/content/drive")

In [ ]:
# ---- Pinned libraries (torch comes with Colab and is logged in every *_meta.json) ----
# If a later cell fails with an import error: Runtime > Restart session, then run every cell again except this one.
!pip install -q transformers==4.56.2 trl==0.23.1 peft==0.17.1 accelerate==1.10.1 datasets==4.1.1

In [ ]:
# ---- Hugging Face access ----
# One-time: accept the Gemma licence at https://huggingface.co/google/gemma-3-1b-it
# and add a read token as a Colab secret named HF_TOKEN (key icon on the left).
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# ---- Copy the frozen dataset (v4) to local disk and check it is unchanged ----
import hashlib, shutil
DATA_VERSION = "v4"
DATA_FILES = {   # sha256 from data/v4/MANIFEST.json (frozen 2026-11-26)
    "splits/train.csv": "a6832dc2104f430c684e961c08be6e51f075ead1559d2559905f38b871de1053",
    "splits/test.csv": "6f85e02b5d8e5dc9d90d8eb30137342fd02a3943302fbe286f574474681b6a7c",
    "structured/structured.csv": "e133b681b2e77c467020b58123e6dc60d7fd576ce25cc500c7a290285a43bbc9",
}
for rel, sha in DATA_FILES.items():
    dst = f"{LOCAL}/{rel}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(f"{DRIVE_ROOT}/data/{DATA_VERSION}/{rel}", dst)
    got = hashlib.sha256(open(dst, "rb").read()).hexdigest()
    assert got == sha, f"STOP: {rel} differs from the frozen {DATA_VERSION} file"
print("dataset OK:", DATA_VERSION, ", ".join(DATA_FILES))


In [ ]:
# ---- GPU check ----
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), "| dtype:", "bf16" if torch.cuda.is_bf16_supported() else "fp32 (T4: slower, still correct)")

## Code

In [ ]:
# ---- Shared settings: identical for the baseline and the fine-tuned run. Do not edit after the first run. ----
import gc, json, os, platform, re, time
import numpy as np
import pandas as pd

MODEL_ID = "google/gemma-3-1b-it"
SEED = 42
DEV_FRAC = 0.10          # share of TRAIN stories held out for eval loss / checkpoint choice
PROMPT_VERSION = "p2"     # p2 = p1 + the "Bilmiyorum." rule (dataset v4)

# Decoding is greedy and fixed so that baseline and fine-tuned outputs are comparable.
GEN_KWARGS = dict(do_sample=False, max_new_tokens=64, repetition_penalty=1.0)

INSTRUCTION = (
    "Aşağıda bir hikâyenin ortasından alınmış bir bölüm var. "
    "Soru, bu bölümden önce ya da sonra olan bir şeyi soruyor. "
    "Bölümdeki ipuçlarını kullanarak en olası cevabı Türkçe ve tek bir kısa cümleyle yaz. "
    "Bölüm cevabı çıkarmak için yeterli ipucu vermiyorsa yalnızca \"Bilmiyorum.\" yaz."
)

# Abstention = the answer says the text does not give the information. Same pattern in notebooks 01, 02 and 03.
# The pattern is a first pass only: in notebook 03 the judge also flags abstentions written in other words.
# Word boundaries keep story answers such as "bilmiyordu" (he did not know) from counting as abstention.
ABSTAIN = "Bilmiyorum."
_ABST = re.compile(
    r"\b(bilmiyorum|bilemiyorum|bilemem|bilinmiyor|bilinemez|belirtilmemiş|belirtilmiyor|söylenmemiş|söylenmiyor"
    r"|bahsedilmiyor|bahsedilmemiş|söz edilmiyor|söz edilmemiş|cevap veremem|çıkarılamaz|çıkarılamıyor|tahmin edilemez)\b"
    r"|bilgi (yok|verilmemiş|verilmiyor|bulunmuyor|yer almıyor)|yeterli (bilgi|ipucu)"
    r"|(metinde|metinden|hikâyede|hikayede|bölümde|bölümden) (yer almıyor|geçmiyor|anlaşılmıyor|yok)\b|\bbelli değil\b")


def tr_lower(s):
    return str(s).replace("I", "ı").replace("İ", "i").lower()


def is_abstention(text):
    return bool(_ABST.search(tr_lower(text)))


def build_messages(part2, question):
    """Single user turn (Gemma 3 has no system role)."""
    return [{"role": "user", "content": f"{INSTRUCTION}\n\nMetin: {part2}\n\nSoru: {question}"}]


def load_split(data_dir, split):
    df = pd.read_csv(os.path.join(data_dir, "splits", f"{split}.csv"))
    need = {"qid", "text_id", "direction", "part2", "question", "answer", "answerable"}
    missing = need - set(df.columns)
    assert not missing, f"{split}.csv is missing columns {missing} (dataset v4 or later needed)"
    assert df[list(need)].notna().all().all(), f"{split}.csv has empty cells"
    assert df.qid.is_unique, f"{split}.csv has duplicate qid"
    return df


def train_dev_split(train_df, dev_frac=DEV_FRAC, seed=SEED):
    """Hold out whole stories from train (never from test)."""
    ids = np.array(sorted(train_df.text_id.unique()))
    perm = np.random.default_rng(seed).permutation(len(ids))
    n_dev = int(round(dev_frac * len(ids)))
    dev_ids = set(ids[perm[:n_dev]])
    dev = train_df[train_df.text_id.isin(dev_ids)].reset_index(drop=True)
    tr = train_df[~train_df.text_id.isin(dev_ids)].reset_index(drop=True)
    assert not set(tr.text_id) & set(dev.text_id)
    return tr, dev


def to_prompt_completion(df):
    """Conversational prompt-completion format: TRL computes the loss on the completion only."""
    from datasets import Dataset
    return Dataset.from_list([
        {"prompt": build_messages(r.part2, r.question),
         "completion": [{"role": "assistant", "content": r.answer}]}
        for r in df.itertuples()
    ])


def pick_dtype():
    """bf16 on Ampere+ (L4, A100). fp32 on T4: Gemma 3 is unstable in fp16."""
    import torch
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float32


def load_model_and_tokenizer(model_id=MODEL_ID, adapter=None, padding_side="right"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    dtype = pick_dtype()
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = padding_side
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, attn_implementation="eager")
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    return model, tok, dtype


def stop_token_ids(tok):
    ids = {tok.eos_token_id}
    eot = tok.convert_tokens_to_ids("<end_of_turn>")
    if eot is not None and eot != tok.unk_token_id:
        ids.add(eot)
    return sorted(i for i in ids if i is not None)


def env_info():
    import torch, transformers, trl, peft
    return {
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "trl": trl.__version__, "peft": peft.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }


def write_json(obj, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def update_json(path, **fields):
    """Add or replace top-level fields in a JSON file (created if missing)."""
    obj = json.load(open(path, encoding="utf-8")) if os.path.exists(path) else {}
    obj.update(fields)
    write_json(obj, path)


# Chart colors: categorical slots 1/2 for the two systems, one blue ramp for ordered levels.
C_BASE, C_FT = "#2a78d6", "#eb6834"
C_TEXT, C_MUTED, C_GRID = "#0b0b0b", "#52514e", "#e4e3df"


def style_axes(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(C_MUTED)
    ax.tick_params(colors=C_MUTED, labelsize=9)
    ax.grid(axis="y", color=C_GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def free_gpu():
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ---- Answer a split with the base model, or base model + LoRA adapter ----
def clean(text):
    """Keep the first non-empty line; the task asks for one short sentence."""
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    return lines[0] if lines else ""


def predict(split, system, out, adapter=None, batch_size=16, limit=0, data_dir=LOCAL, model_id=MODEL_ID,
            meta_path=None, meta_key="test_predictions"):
    """Writes the predictions CSV; run metadata is stored under meta_key in meta_path (if given)."""
    import torch
    torch.manual_seed(SEED)
    df = load_split(data_dir, split)
    if limit:
        df = df.head(limit)
    model, tok, dtype = load_model_and_tokenizer(model_id, adapter, padding_side="left")
    model.eval()
    stops = stop_token_ids(tok)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

    raw, t0 = [], time.time()
    for i in range(0, len(df), batch_size):
        chunk = df.iloc[i:i + batch_size]
        texts = [tok.apply_chat_template(build_messages(r.part2, r.question), tokenize=False,
                                         add_generation_prompt=True) for r in chunk.itertuples()]
        # the chat template already contains <bos>; do not add it twice
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        enc = {k: enc[k] for k in ("input_ids", "attention_mask")}
        with torch.no_grad():
            gen = model.generate(**enc, **GEN_KWARGS, eos_token_id=stops, pad_token_id=pad_id)
        raw += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"{min(i + batch_size, len(df))}/{len(df)}", flush=True)

    keep = ["qid", "text_id", "direction", "question", "answer", "answerable", "source", "in_v3_test"]
    res = df[[c for c in keep if c in df]].copy()
    res["prediction"] = [clean(t) for t in raw]
    res["raw_output"] = raw
    res["system"] = system
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    res.to_csv(out, index=False)
    meta = {
        "system": system, "split": split, "n": len(res), "file": os.path.basename(out), "model_id": model_id,
        "adapter": adapter, "prompt_version": PROMPT_VERSION, "gen_kwargs": GEN_KWARGS, "dtype": str(dtype),
        "seconds": round(time.time() - t0, 1), "empty_predictions": int((res.prediction == "").sum()),
        "abstained": int(res.prediction.map(is_abstention).sum()),
        "env": env_info(),
    }
    if meta_path:
        update_json(meta_path, **{meta_key: meta})
    del model
    free_gpu()
    print(f"saved {len(res)} predictions to {out} | empty: {(res.prediction == '').sum()}")
    return res

def abstention_check(res):
    """Quick look before the judge: how often the model said "Bilmiyorum." on each kind of item."""
    a, u = res.prediction.map(is_abstention), res.answerable == 0
    print(f"abstained on {a[u].mean():.1%} of {u.sum()} Bilmiyorum items and {a[~u].mean():.1%} of {(~u).sum()} answerable items")
    if "source" in res:
        print(res[u].assign(abstained=a[u]).groupby("source").abstained.mean().round(3).to_string())


In [ ]:
# ---- LoRA SFT of gemma-3-1b-it on (part2, question) -> answer, resumable ----
import math, re, shutil
from transformers import TrainerCallback


def show_one_example(trainer, tok):
    """Print what the model sees and which tokens carry loss, so masking can be checked by eye."""
    ex = trainer.train_dataset[0]
    ids, mask = ex["input_ids"], ex.get("completion_mask")
    print("---- model input ----")
    print(tok.decode(ids))
    if mask is not None:
        print("---- tokens with loss (should be: <start_of_turn>model + answer + <end_of_turn>) ----")
        print(tok.decode([t for t, m in zip(ids, mask) if m]))
    print("---------------------")


def plot_loss(log_history, path):
    import matplotlib.pyplot as plt
    h = pd.DataFrame(log_history)
    tr = h.dropna(subset=["loss"]) if "loss" in h else h.iloc[:0]
    ev = h.dropna(subset=["eval_loss"]) if "eval_loss" in h else h.iloc[:0]
    fig, ax = plt.subplots(figsize=(6.4, 3.6), dpi=150)
    ax.plot(tr.step, tr.loss, color=C_BASE, linewidth=2, label="train")
    if len(ev):
        ax.plot(ev.step, ev.eval_loss, color=C_FT, linewidth=2, marker="o", markersize=7,
                markeredgecolor="white", markeredgewidth=2, label="dev")
        best = ev.loc[ev.eval_loss.idxmin()]
        ax.annotate(f"best dev {best.eval_loss:.3f}", (best.step, best.eval_loss), xytext=(0, 10),
                    textcoords="offset points", ha="center", fontsize=8, color=C_TEXT)
    style_axes(ax)
    ax.set_xlabel("optimizer step", color=C_MUTED, fontsize=9)
    ax.set_ylabel("loss (answer tokens)", color=C_MUTED, fontsize=9)
    ax.set_title("Training and dev loss", loc="left", fontsize=11, color=C_TEXT)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(path)
    plt.show()


def subsample_stories(df, pct, seed=SEED):
    """Keep pct % of the STORIES in df (with all their pairs). Nested across pct values for a fixed seed."""
    assert 0 < pct <= 100, "TRAIN_PCT must be in (0, 100]"
    if pct == 100:
        return df
    ids = np.array(sorted(df.text_id.unique()))
    perm = np.random.default_rng(seed + 1).permutation(len(ids))
    keep = set(ids[perm[:max(1, int(round(pct / 100 * len(ids))))]])
    return df[df.text_id.isin(keep)].reset_index(drop=True)


_CKPT = re.compile(r"^checkpoint-(\d+)$")


def list_ckpts(d):
    """Complete checkpoints in d, oldest first. A folder counts only if its key files are all there."""
    if not os.path.isdir(d):
        return []
    out = []
    for n in os.listdir(d):
        m = _CKPT.match(n)
        if m and all(os.path.exists(f"{d}/{n}/{f}") for f in
                     ("trainer_state.json", "optimizer.pt", "scheduler.pt", "adapter_model.safetensors")):
            out.append((int(m.group(1)), n))
    return [n for _, n in sorted(out)]


class DriveSync(TrainerCallback):
    """After every save, mirror the kept local checkpoints to Drive (copy to *.tmp, then rename: a cut-off copy is never used)."""
    def __init__(self, local, remote):
        self.local, self.remote = local, remote

    def on_save(self, args, state, control, **kwargs):
        os.makedirs(self.remote, exist_ok=True)
        keep = list_ckpts(self.local)
        for n in keep:
            dst = f"{self.remote}/{n}"
            if not os.path.exists(dst):
                tmp = dst + ".tmp"
                shutil.rmtree(tmp, ignore_errors=True)
                shutil.copytree(f"{self.local}/{n}", tmp)
                os.rename(tmp, dst)
        for n in os.listdir(self.remote):
            if n not in keep:
                shutil.rmtree(f"{self.remote}/{n}", ignore_errors=True)
        print(f"[drive] step {state.global_step}: saved {keep}", flush=True)


def restore_from_drive(local, remote):
    """New session: the local disk is empty, so copy the Drive checkpoints back. Returns the one to resume from."""
    if not list_ckpts(local):
        for n in list_ckpts(remote):
            shutil.copytree(f"{remote}/{n}", f"{local}/{n}", dirs_exist_ok=True)
    found = list_ckpts(local)
    return f"{local}/{found[-1]}" if found else None


def train(out_dir, epochs=3, lr=2e-4, lora_r=16, lora_alpha=32, lora_dropout=0.05, batch_size=8, grad_accum=2,
          max_length=512, dev_frac=DEV_FRAC, train_pct=100, save_every=150, max_steps=-1, seed=SEED,
          data_dir=LOCAL, model_id=MODEL_ID, ckpt_local=CKPT_LOCAL, ckpt_drive=CKPT_DRIVE):
    """Saves to out_dir: adapter/ (LoRA weights + config), run.json, loss.png.
    Resumes from the last checkpoint in ckpt_drive/<run name> if the same run was cut off."""
    args = json.loads(json.dumps(dict(locals()), default=str))
    import torch
    from peft import LoraConfig
    from transformers import set_seed
    from trl import SFTConfig, SFTTrainer

    smoke = max_steps > 0
    name = os.path.basename(os.path.normpath(out_dir))
    local, remote = f"{ckpt_local}/{name}", f"{ckpt_drive}/{name}"
    run_json = f"{out_dir}/run.json"

    # ---- already finished? ----
    if not smoke and os.path.exists(f"{out_dir}/adapter/adapter_model.safetensors") and os.path.exists(run_json) \
            and "best_dev_loss" in json.load(open(run_json, encoding="utf-8")):
        print(f"training already finished for {name}: skipping (delete {out_dir} to train again)")
        return

    # ---- same settings as the cut-off run? ----
    if smoke:
        shutil.rmtree(local, ignore_errors=True)       # the smoke test always starts fresh
        resume = None
    else:
        os.makedirs(out_dir, exist_ok=True)
        prev = json.load(open(run_json, encoding="utf-8")).get("args") if os.path.exists(run_json) else None
        if prev is not None:
            diff = {k: (prev.get(k), v) for k, v in args.items() if prev.get(k) != v}
            assert not diff, f"STOP: {name} was started with other settings {diff}. Use the old settings or a new RUN_ID."
        else:
            update_json(run_json, args=args, status="training")
        resume = restore_from_drive(local, remote)

    set_seed(seed)
    tr_df, dev_df = train_dev_split(load_split(data_dir, "train"), dev_frac, seed)   # dev fixed for every train_pct
    n_full = len(tr_df)
    tr_df = subsample_stories(tr_df, train_pct, seed)
    steps_per_epoch = math.ceil(math.ceil(len(tr_df) / batch_size) / grad_accum)
    every = max_steps if smoke else max(1, min(save_every, steps_per_epoch))
    print(f"train {len(tr_df)} items / {tr_df.text_id.nunique()} stories ({train_pct}% of {n_full} items), "
          f"{(tr_df.answerable == 0).mean():.1%} Bilmiyorum | "
          f"dev {len(dev_df)} items / {dev_df.text_id.nunique()} stories, {(dev_df.answerable == 0).mean():.1%} Bilmiyorum")
    print(f"{steps_per_epoch} steps/epoch x {epochs} epochs = {steps_per_epoch * epochs} steps | eval + save every {every} | "
          + (f"RESUMING from {os.path.basename(resume)}" if resume else "starting from step 0"))

    model, tok, dtype = load_model_and_tokenizer(model_id, padding_side="right")
    model.config.use_cache = False
    cfg = SFTConfig(
        output_dir=local, num_train_epochs=epochs, max_steps=max_steps,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum, learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, weight_decay=0.0, logging_steps=1 if smoke else 10,
        eval_strategy="steps", save_strategy="steps", eval_steps=every, save_steps=every,
        save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        bf16=(dtype == torch.bfloat16), fp16=False, max_length=max_length, packing=False,
        completion_only_loss=True, report_to="none", seed=seed,
    )
    peft_cfg = LoraConfig(r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                          target_modules="all-linear", task_type="CAUSAL_LM")
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=to_prompt_completion(tr_df),
                         eval_dataset=to_prompt_completion(dev_df), processing_class=tok, peft_config=peft_cfg,
                         callbacks=[] if smoke else [DriveSync(local, remote)])
    trainer.model.print_trainable_parameters()
    show_one_example(trainer, tok)

    trainer.train(resume_from_checkpoint=resume)   # restores weights, optimizer, LR schedule, step and data order
    final_eval = trainer.evaluate()
    trainer.save_model(f"{out_dir}/adapter")      # LoRA weights only (best checkpoint by dev loss)
    for f in os.listdir(f"{out_dir}/adapter"):     # keep weights + config; tokenizer always comes from MODEL_ID
        if f not in ("adapter_model.safetensors", "adapter_config.json"):
            os.remove(f"{out_dir}/adapter/{f}")
    update_json(run_json,
        args=args, status="trained", model_id=model_id, prompt_version=PROMPT_VERSION, dtype=str(dtype),
        data_version=globals().get("DATA_VERSION"), train_pct=train_pct,
        n_train_pairs=len(tr_df), n_train_stories=int(tr_df.text_id.nunique()), n_train_pairs_full=n_full,
        n_dev_pairs=len(dev_df), n_train_bilmiyorum=int((tr_df.answerable == 0).sum()),
        n_dev_bilmiyorum=int((dev_df.answerable == 0).sum()), resumed_from=os.path.basename(resume) if resume else None,
        best_checkpoint=trainer.state.best_model_checkpoint, best_dev_loss=trainer.state.best_metric,
        final_eval=final_eval, env=env_info(),
        dev_text_ids=sorted(dev_df.text_id.unique().tolist()),
        train_text_ids=sorted(tr_df.text_id.unique().tolist()) if train_pct < 100 else "all",
        log_history=trainer.state.log_history)
    plot_loss(trainer.state.log_history, f"{out_dir}/loss.png")
    print(f"adapter saved to {out_dir}/adapter | best dev loss {trainer.state.best_metric}")
    del trainer, model
    free_gpu()
    shutil.rmtree(local, ignore_errors=True)
    if not smoke:
        shutil.rmtree(remote, ignore_errors=True)  # adapter is safe in out_dir; checkpoints no longer needed
        print(f"deleted checkpoints in {remote}")

## Smoke test (5 steps)
Read the printed example: the "tokens with loss" block must contain only `<start_of_turn>model`, the answer and `<end_of_turn>`. The story and the question must not appear there.
The smoke test never touches Drive checkpoints. Skip it when resuming.

In [ ]:
train("/content/smoke_run", max_steps=5)

## Full training
Set `TRAIN_PCT` and the other settings, then run. After a disconnect: Setup cells, Code cells, then this cell again without changes.

In [ ]:
# ---- Full training: edit these, then run. To resume after a disconnect, run this cell again UNCHANGED. ----
TRAIN_PCT = 100          # share of train stories to use, 1..100 (dev set is the same for every value)
RUN_DATE = "2026-11-26"  # fixed string (not today's date) so a resumed session finds the same run
LR, R, EPOCHS = 2e-4, 16, 3
SAVE_EVERY = 150         # optimizer steps between checkpoints (= the most work a disconnect can cost)

RUN_ID = f"{RUN_DATE}_{DATA_VERSION}_{TRAIN_PCT}pct_lora_r{R}_lr{LR:g}_ep{EPOCHS}"
RUN = f"{DRIVE_ROOT}/runs/{RUN_ID}"
print(RUN)
train(RUN, epochs=EPOCHS, lr=LR, lora_r=R, lora_alpha=2 * R, train_pct=TRAIN_PCT, save_every=SAVE_EVERY)

## Test predictions with the trained adapter

In [ ]:
pred_path = f"{RUN}/predictions_test.csv"
done = os.path.exists(pred_path) and "test_predictions" in json.load(open(f"{RUN}/run.json", encoding="utf-8"))
if done:
    print("test predictions already saved, skipping:", pred_path)
    p = pd.read_csv(pred_path, keep_default_na=False)
else:
    p = predict("test", "finetuned", pred_path, adapter=f"{RUN}/adapter", meta_path=f"{RUN}/run.json")
abstention_check(p)
p.sample(10, random_state=0)[["question", "answer", "prediction"]]

## Optional: the v3 adapter on the v4 test set
The adapter trained on v3 (no "Bilmiyorum." items) answers the same 668 items with the same prompt `p2`. Comparing it with the v4 run isolates the effect of the "Bilmiyorum." training data. In notebook 03 it appears as `finetuned_v3`.

In [ ]:
V3_RUN_ID = "PASTE_V3_RUN_ID_HERE"     # folder name under runs/ of the v3 run (100%)
V3_OUT = f"{DRIVE_ROOT}/eval/finetuned_v3_on_v4"
if os.path.exists(f"{V3_OUT}/predictions_test.csv"):
    print("already saved, skipping:", f"{V3_OUT}/predictions_test.csv")
    p3 = pd.read_csv(f"{V3_OUT}/predictions_test.csv", keep_default_na=False)
else:
    p3 = predict("test", "finetuned_v3", f"{V3_OUT}/predictions_test.csv",
                 adapter=f"{DRIVE_ROOT}/runs/{V3_RUN_ID}/adapter", meta_path=f"{V3_OUT}/run.json")
abstention_check(p3)

## Optional: compact a run made with the old notebook
Old runs have `checkpoints/`, `dev_ids.csv`, `log_history.csv`, `run_config.json`, `eval/` and tokenizer files in `adapter/`.
This merges the small files into `run.json`, draws `loss.png`, moves the predictions up one level and deletes the rest.
No GPU needed: run the Settings, Mount and Code cells first, then this one.

In [ ]:
import shutil

def compact_run(run):
    load = lambda f: json.load(open(f, encoding="utf-8"))
    cfg = load(f"{run}/run_config.json") if os.path.exists(f"{run}/run_config.json") else {}
    if os.path.exists(f"{run}/dev_ids.csv"):
        cfg["dev_text_ids"] = pd.read_csv(f"{run}/dev_ids.csv").text_id.tolist()
    if os.path.exists(f"{run}/log_history.csv"):
        h = pd.read_csv(f"{run}/log_history.csv")
        cfg["log_history"] = [{k: v for k, v in r.items() if pd.notna(v)} for r in h.to_dict("records")]
    if os.path.exists(f"{run}/eval/predictions_test_meta.json"):
        cfg["test_predictions"] = load(f"{run}/eval/predictions_test_meta.json")
    write_json(cfg, f"{run}/run.json")
    if "log_history" in cfg:
        plot_loss(cfg["log_history"], f"{run}/loss.png")
    if os.path.exists(f"{run}/eval/predictions_test.csv"):
        shutil.move(f"{run}/eval/predictions_test.csv", f"{run}/predictions_test.csv")
    for f in ["run_config.json", "dev_ids.csv", "log_history.csv"]:
        if os.path.exists(f"{run}/{f}"):
            os.remove(f"{run}/{f}")
    for d in ["checkpoints", "eval"]:
        shutil.rmtree(f"{run}/{d}", ignore_errors=True)
    for f in os.listdir(f"{run}/adapter"):
        if f not in ("adapter_model.safetensors", "adapter_config.json"):
            os.remove(f"{run}/adapter/{f}")
    for root, _, files in os.walk(run):
        for f in sorted(files):
            print(os.path.relpath(os.path.join(root, f), run))

# compact_run(f"{DRIVE_ROOT}/runs/PASTE_RUN_ID_HERE")